<a href="https://colab.research.google.com/github/irthifa/sql-data-validation-northwind/blob/main/SQL_data_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SQL Data Validation
Dataset: Northwind Retail Database (sample dataset)

---


Goal: Validate data integrity (duplicates, foreign keys) and generate summary reports.

---


Outcome: Cleaned data exported for Tableau dashboards.

In [ ]:
# Install SQLite support
import sqlite3
import pandas as pd

# Connect to in-memory SQLite database
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

In [ ]:
import pandas as pd
import sqlite3

# Download CSVs from GitHub
!wget https://raw.githubusercontent.com/neo4j-graph-examples/northwind/main/import/customers.csv -O customers.csv
!wget https://raw.githubusercontent.com/neo4j-graph-examples/northwind/main/import/orders.csv -O orders.csv
!wget https://raw.githubusercontent.com/neo4j-graph-examples/northwind/main/import/order-details.csv -O order_details.csv

# Load into Pandas
customers = pd.read_csv("customers.csv")
orders = pd.read_csv("orders.csv")
order_details = pd.read_csv("order_details.csv")

# Write to SQLite
conn = sqlite3.connect(":memory:")
customers.to_sql("Customers", conn, index=False, if_exists="replace")
orders.to_sql("Orders", conn, index=False, if_exists="replace")
order_details.to_sql("OrderDetails", conn, index=False, if_exists="replace")

# Test query
pd.read_sql("SELECT * FROM Customers LIMIT 5;", conn)


--2026-06-23 20:02:42--  https://raw.githubusercontent.com/neo4j-graph-examples/northwind/main/import/customers.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 11889 (12K) [text/plain]
Saving to: ‘customers.csv’

customers.csv       100%[===================>]  11.61K  --.-KB/s    in 0s      

2026-06-23 20:02:42 (86.1 MB/s) - ‘customers.csv’ saved [11889/11889]

--2026-06-23 20:02:42--  https://raw.githubusercontent.com/neo4j-graph-examples/northwind/main/import/orders.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 132885

,customerID,companyName,contactName,contactTitle,address,city,region,postalCode,country,phone,fax
0,ALFKI,Alfreds Futterkiste,Maria Anders,Sales Representative,Obere Str. 57,Berlin,None,12209,Germany,030-0074321,030-0076545
1,ANATR,Ana Trujillo Emparedados y helados,Ana Trujillo,Owner,Avda. de la Constitución 2222,México D.F.,None,05021,Mexico,(5) 555-4729,(5) 555-3745
2,ANTON,Antonio Moreno Taquería,Antonio Moreno,Owner,Mataderos 2312,México D.F.,None,05023,Mexico,(5) 555-3932,None
3,AROUT,Around the Horn,Thomas Hardy,Sales Representative,120 Hanover Sq.,London,None,WA1 1DP,UK,(171) 555-7788,(171) 555-6750
4,BERGS,Berglunds snabbköp,Christina Berglund,Order Administrator,Berguvsvägen 8,Luleå,None,S-958 22,Sweden,0921-12 34 65,0921-12 34 67


In [ ]:
#Check for duplicates
query = """
SELECT ContactName, COUNT(*) AS cnt
FROM Customers
GROUP BY ContactName
HAVING COUNT(*) > 1;
"""
pd.read_sql(query, conn)

,contactName,cnt


In [ ]:
#Validate foreign keys
query = """
SELECT o.OrderID, o.CustomerID
FROM Orders o
LEFT JOIN Customers c ON o.CustomerID = c.CustomerID
WHERE c.CustomerID IS NULL;
"""
pd.read_sql(query, conn)

,orderID,customerID


In [ ]:
# Total sales per region
query1 = """
SELECT c.Country, SUM(od.UnitPrice * od.Quantity) AS TotalSales
FROM Orders o
JOIN Customers c ON o.CustomerID = c.CustomerID
JOIN OrderDetails od ON o.OrderID = od.OrderID
GROUP BY c.Country
ORDER BY TotalSales DESC;
"""
sales_by_region = pd.read_sql(query1, conn)
sales_by_region.head()

# Top 10 customers
query2 = """
SELECT c.ContactName, SUM(od.UnitPrice * od.Quantity) AS TotalSpent
FROM Customers c
JOIN Orders o ON c.CustomerID = o.CustomerID
JOIN OrderDetails od ON o.OrderID = od.OrderID
GROUP BY c.ContactName
ORDER BY TotalSpent DESC
LIMIT 10;
"""
top_customers = pd.read_sql(query2, conn)
top_customers

,contactName,TotalSpent
0,Horst Kloss,117483.39
1,Jose Pavarotti,115673.39
2,Roland Mendel,113236.68
3,Patricia McKenna,57317.39
4,Paula Wilson,52245.90
5,Mario Pontes,34101.15
6,Maria Larsson,32555.55
7,Jean Fresnière,32203.90
8,Philip Cramer,31745.75
9,Lúcia Carvalho,30226.10


In [ ]:
# Save outputs as CSV for Tableau
sales_by_region.to_csv("sales_by_region.csv", index=False)
top_customers.to_csv("top_customers.csv", index=False)

print("CSV files ready for Tableau import!")

CSV files ready for Tableau import!


In [ ]:
#Download CSV
from google.colab import files
files.download("sales_by_region.csv")
files.download("top_customers.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>